# 8.4 Systematic Model Comparison Framework

## Course 3: Advanced Classification Models for Student Success

This notebook extends **4.1 Systematic Model Comparison** by comparing the original three Course 3 models against a fourth model: the **survey-enhanced XGBoost model** introduced in Module 8.

### Models compared

To predict student outcomes, the methodology progresses through five distinct machine learning configurations. First, it establishes baselines using Regularized Logistic Regression, Decision Trees, and Random Forest driven exclusively by administrative and student-record data. Next, it elevates performance utilizing standard XGBoost on the same baseline features, before culminating in a Survey-Enhanced XGBoost model. This final, most robust framework integrates the comprehensive survey master matrix—incorporating academic metrics, demographics, self-reported survey responses, and compressed text features—to maximize predictive accuracy.

The goal is not only to determine which model performs best, but to create a reusable framework for comparing model performance, interpretability, deployment feasibility, and institutional usefulness.

### Learning Objectives

1. Train and evaluate all three models on the same dataset with optimized hyperparameters
2. Evaluate model performance using a comprehensive suite of metrics, including Accuracy, Precision, Recall, F1 Score, ROC-AUC, and calibration metrics, while understanding their relevance in a higher education context.
3. Analyze trade-offs between predictive performance, interpretability, deployment feasibility, and institutional usefulness to make informed model recommendations.
4. Interpret model explainability through feature importance analysis, including individual features and feature groups, to understand model drivers.
5. Assess and compare model consistency and fairness across diverse demographic subgroups to identify potential biases and ensure equitable outcomes.

## 1. Setup

Expected files:

- `training.csv`
- `testing.csv`
- `ML_SURVEY_MASTER_TRAIN.csv`
- `ML_SURVEY_MASTER_TEST.csv`

In Google Colab, update `filepath` to the folder where these files live.

In [2]:
90_000+32


90032

In [ ]:
# If needed in Colab, uncomment the next line:
# !pip -q install xgboost

import numpy as np
import pandas as pd
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss, classification_report
)
import pickle

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the Survey Enhanced Data



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'

train_df = pd.read_csv(filepath + 'ML_SURVEY_MASTER_TRAIN.csv')
test_df = pd.read_csv(filepath + 'ML_SURVEY_MASTER_TEST.csv')

print("Training shape:", train_df.shape)
print("Testing shape: ", test_df.shape)

train_df.head()

## 3. Create Administrative Data Objects for the Original Models

Select the non-text variables from the survey-enhanced dataframe. These are the same administrative/student-record features used in the original **4.1 Systematic Model Comparison** notebook.

In [ ]:
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = [
    'HS_GPA', 'HS_MATH_GPA', 'HS_ENGL_GPA',
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1', 'UNITS_COMPLETED_2',
    'DFW_UNITS_1', 'DFW_UNITS_2',
    'GPA_1', 'GPA_2',
    'DFW_RATE_1', 'DFW_RATE_2',
    'GRADE_POINTS_1', 'GRADE_POINTS_2'
]

categorical_features = ['RACE_ETHNICITY', 'GENDER', 'FIRST_GEN_STATUS', 'COLLEGE']

numeric_features = [c for c in numeric_features if c in train_df.columns]
categorical_features = [c for c in categorical_features if c in train_df.columns]

train_enc = pd.get_dummies(
    train_df[numeric_features + categorical_features],
    columns=categorical_features,
    drop_first=True
)

test_enc = pd.get_dummies(
    test_df[numeric_features + categorical_features],
    columns=categorical_features,
    drop_first=True
)

train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)
train_enc = train_enc.fillna(train_enc.median())
test_enc = test_enc.fillna(test_enc.median())

X_train_admin = train_enc
y_train = train_df['DEPARTED']
X_test_admin = test_enc
y_test = test_df['DEPARTED']

scaler = StandardScaler()
X_train_admin_scaled = scaler.fit_transform(X_train_admin)
X_test_admin_scaled = scaler.transform(X_test_admin)

print(f"Administrative training rows: {X_train_admin.shape[0]:,}")
print(f"Administrative testing rows:  {X_test_admin.shape[0]:,}")
print(f"Administrative features:      {X_train_admin.shape[1]:,}")
print(f"Departure rate: train={y_train.mean():.2%}, test={y_test.mean():.2%}")

## 4. Load Survey-Enhanced Master Matrix

The survey-enhanced model uses the Module 8 master matrix. This matrix already combines academic, demographic, survey, and text-derived features such as `TEXT_PC1`, `TEXT_PC2`, and so on.

In [ ]:
X_train_admin = train_df[numeric_features + categorical_features]
X_test_admin = test_df[numeric_features + categorical_features]

X_train_survey = train_df.drop(columns=['DEPARTED', 'SEM_3_STATUS'])
X_test_survey = test_df.drop(columns=['DEPARTED', 'SEM_3_STATUS'])

X_train_survey.columns = X_train_survey.columns.astype(str)
X_test_survey.columns = X_test_survey.columns.astype(str)
X_train_survey, X_test_survey = X_train_survey.align(X_test_survey, join='left', axis=1, fill_value=0)

X_train_survey = X_train_survey.fillna(X_train_survey.median(numeric_only=True))
X_test_survey = X_test_survey.fillna(X_test_survey.median(numeric_only=True))

print(f"Survey-enhanced training rows: {X_train_survey.shape[0]:,}")
print(f"Survey-enhanced testing rows:  {X_test_survey.shape[0]:,}")
print(f"Survey-enhanced features:      {X_train_survey.shape[1]:,}")
print(f"Departure rate: train={y_train.mean():.2%}, test={y_test.mean():.2%}")

## 5. Model Hyperparameter Registry

This table records the exact model configurations used in the comparison.

The first three models come from the original 4.1 notebook. The fourth model comes from the Module 8 survey-enhanced XGBoost workflow.

In [ ]:
# Load the model from the pickle file
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'
filename1 = f'{model_filepath}dt_tuned.pkl'
dt = pickle.load(open(filename1, 'rb'))

filename1 = f'{model_filepath}rf_tuned.pkl'
rf = pickle.load(open(filename1, 'rb'))

filename1 = f'{model_filepath}xgb_tuned.pkl'
xgb_admin = pickle.load(open(filename1, 'rb'))

filename1 = f'{model_filepath}xgb_tuned.pkl'
xgb_survey = pickle.load(open(filename1, 'rb'))

lr = LogisticRegression(penalty='l1', solver='saga', C=0.01, max_iter=1000, random_state=RANDOM_STATE)

## 6. Train the Four Models

The first three models are trained on the original administrative feature matrix. The survey-enhanced model is trained on the expanded Module 8 master matrix.

In [ ]:

trained_models = {}

start = time.time()
lr.fit(X_train_admin, y_train)
trained_models['Regularized Logistic'] = {
    'model': lr, 'X_test': X_test_admin, 'y_test': y_test,
    'train_time': time.time() - start, 'feature_set': 'Administrative'
}

start = time.time()
dt.fit(X_train_admin, y_train)
trained_models['Decision Tree'] = {
    'model': dt, 'X_test': X_test_admin, 'y_test': y_test,
    'train_time': time.time() - start, 'feature_set': 'Administrative'
}

start = time.time()

rf.fit(X_train_admin, y_train)
trained_models['Random Forest'] = {
    'model': rf, 'X_test': X_test_admin, 'y_test': y_test,
    'train_time': time.time() - start, 'feature_set': 'Administrative'
}

start = time.time()

xgb_admin.fit(X_train_admin, y_train)
trained_models['XGBoost'] = {
    'model': xgb_admin, 'X_test': X_test_admin, 'y_test': y_test,
    'train_time': time.time() - start, 'feature_set': 'Administrative'
}

start = time.time()

xgb_survey.fit(X_train_survey, y_train)
trained_models['Survey-Enhanced XGBoost'] = {
    'model': xgb_survey, 'X_test': X_test_survey, 'y_test': y_test,
    'train_time': time.time() - start, 'feature_set': 'Survey-enhanced'
}

print('All four models trained successfully.')


## 7. Evaluate Model Performance

The comparison uses the same core metrics as the original 4.1 notebook: Accuracy, Precision, Recall, F1 Score, ROC-AUC, Average Precision, Brier Score, Log Loss, and Training Time.

For student success analytics, pay special attention to **Recall**, **F1 Score**, **Average Precision**, and calibration-related metrics such as **Brier Score** and **Log Loss**.

In [ ]:
def evaluate_binary_classifier(name, model_info):
    model = model_info['model']
    X_test = model_info['X_test']
    y_test = model_info['y_test']
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        'Model': name,
        'Feature Set': model_info['feature_set'],
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1 Score': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob),
        'Avg Precision': average_precision_score(y_test, prob),
        'Brier Score': brier_score_loss(y_test, prob),
        'Log Loss': log_loss(y_test, prob),
        'Train Time (s)': model_info['train_time']
    }

results = [evaluate_binary_classifier(name, info) for name, info in trained_models.items()]
results_df = pd.DataFrame(results).set_index('Model')

print('=' * 110)
print('SYSTEMATIC MODEL COMPARISON: ADMINISTRATIVE VS. SURVEY-ENHANCED MODELS')
print('=' * 110)
print(results_df.round(4).to_string())
print('=' * 110)

results_df.round(4)

## 8. Performance Bar Chart

This view makes the core model trade-offs visible across the primary evaluation metrics.

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
colors = ['#9b59b6', '#3498db', '#e74c3c', '#2ecc71', '#ffa500']

fig = go.Figure()
for i, model_name in enumerate(results_df.index):
    fig.add_trace(go.Bar(
        name=model_name,
        x=metrics,
        y=[results_df.loc[model_name, m] for m in metrics],
        marker_color=colors[i]
    ))

fig.update_layout(
    title='Systematic Model Comparison: Original 4.1 Models + Survey-Enhanced XGBoost',
    barmode='group',
    height=500,
    yaxis_title='Score',
    yaxis=dict(range=[0, 1])
)
fig.show()

## 9. ROC and Precision-Recall Curves

ROC curves show ranking quality across thresholds. Precision-Recall curves are especially useful when the positive class is relatively uncommon, as is often the case with departure or non-enrollment outcomes.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('ROC Curves', 'Precision-Recall Curves'))

for i, (name, info) in enumerate(trained_models.items()):
    model = info['model']
    X_test = info['X_test']
    y_test = info['y_test']
    prob = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             name=f'{name} (AUC={auc:.3f})',
                             line=dict(color=colors[i], width=2)), row=1, col=1)

    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines',
                             name=f'{name} (AP={ap:.3f})',
                             line=dict(color=colors[i], width=2, dash='dot'),
                             showlegend=False), row=1, col=2)

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         line=dict(color='gray', dash='dash'), showlegend=False), row=1, col=1)

fig.update_layout(height=500, title_text='ROC and Precision-Recall Curves')
fig.update_xaxes(title_text='False Positive Rate', row=1, col=1)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2)
fig.update_yaxes(title_text='Precision', row=1, col=2)
fig.show()

## 10. Confusion Matrices

Confusion matrices show the concrete count of correct and incorrect predictions. For student departure prediction, the most institutionally important error is often the **false negative**: a student who was predicted to remain enrolled but actually departed.

In [ ]:
for name, info in trained_models.items():
    model = info['model']
    X_test = info['X_test']
    y_test = info['y_test']
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)

    fig = px.imshow(
        cm,
        text_auto=True,
        labels=dict(x='Predicted Label', y='True Label', color='Count'),
        x=['Predicted Enrolled', 'Predicted Departed'],
        y=['Actual Enrolled', 'Actual Departed'],
        title=f'Confusion Matrix: {name}'
    )
    fig.show()

## 11. Interpretability vs. Performance Trade-off

In 4.1 we used a radar chart to compare not only predictive performance, but also practical institutional considerations. Here we add the survey-enhanced model.

These non-performance ratings are intentionally judgment-based. Adjust them if your institution has different priorities or technical capacity.

In [ ]:
categories = ['ROC-AUC', 'Interpretability', 'Training Speed',
              'Handles Non-linearity', 'Ease of Deployment', 'Stakeholder Trust']

radar_scores = {
    'Regularized Logistic': [results_df.loc['Regularized Logistic', 'ROC-AUC'] * 10, 9, 9, 3, 10, 9],
    'Decision Tree': [results_df.loc['Regularized Logistic', 'ROC-AUC'] * 10, 8, 10, 4, 8, 7],
    'Random Forest': [results_df.loc['Random Forest', 'ROC-AUC'] * 10, 5, 6, 9, 7, 6],
    'XGBoost': [results_df.loc['XGBoost', 'ROC-AUC'] * 10, 4, 7, 10, 6, 4],
    'Survey-Enhanced XGBoost': [results_df.loc['Survey-Enhanced XGBoost', 'ROC-AUC'] * 10, 3, 6, 10, 5, 4]
}

fig = go.Figure()
for i, (name, vals) in enumerate(radar_scores.items()):
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=categories + [categories[0]],
        name=name,
        line=dict(color=colors[i], width=2),
        fill='toself',
        opacity=0.25
    ))

fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
                  title='Model Capabilities Radar Chart', height=600)
fig.show()

## 12. Feature Importance for Tree-Based Models

Feature importance helps explain which variables the tree-based models relied on most. These importance scores are useful for interpretation, but they are **not causal evidence**.

In [ ]:
def feature_importance_table(model, feature_names, model_name, top_n=20):
    if not hasattr(model, 'feature_importances_'):
        print(f'{model_name} does not expose feature_importances_.')
        return None

    fi = (pd.DataFrame({'Feature': list(feature_names), 'Importance': model.feature_importances_})
          .sort_values('Importance', ascending=False)
          .reset_index(drop=True))

    display(fi.head(top_n))

    fig = px.bar(fi.head(top_n).iloc[::-1], x='Importance', y='Feature',
                 orientation='h', title=f'Top {top_n} Feature Importances: {model_name}')
    fig.show()
    return fi

rf_fi = feature_importance_table(rf, X_train_admin.columns, 'Random Forest')
xgb_admin_fi = feature_importance_table(xgb_admin, X_train_admin.columns, 'XGBoost')
xgb_survey_fi = feature_importance_table(xgb_survey, X_train_survey.columns, 'Survey-Enhanced XGBoost')

,Feature,Importance
0,GPA_2,0.279298
1,DFW_RATE_1,0.210188
2,GPA_1,0.191786
3,DFW_RATE_2,0.117157
4,UNITS_ATTEMPTED_2,0.101436
5,HS_GPA,0.066642
6,UNITS_ATTEMPTED_1,0.033493


,Feature,Importance
0,DFW_RATE_1,0.396521
1,GPA_2,0.223567
2,GPA_1,0.161879
3,DFW_RATE_2,0.105992
4,UNITS_ATTEMPTED_2,0.062400
5,UNITS_ATTEMPTED_1,0.025627
6,HS_GPA,0.024015


,Feature,Importance
0,DFW_RATE_1,0.286735
1,GPA_2,0.091149
2,GPA_1,0.075762
3,DFW_RATE_2,0.056003
4,UNITS_ATTEMPTED_2,0.029350
5,RACE_ETHNICITY_Asian,0.013635
6,RACE_ETHNICITY_Two or More Races,0.012768
7,RACE_ETHNICITY_White,0.012247
8,FIRST_GEN_STATUS_First Generation,0.012242
9,UNITS_ATTEMPTED_1,0.011893


Let's compare and contrast variable importance for the three models simultaneously.

In [ ]:
top_n = 5

# Get top N features for each model
rf_top_features = rf_fi.head(top_n).set_index('Feature')[['Importance']]
xgb_admin_top_features = xgb_admin_fi.head(top_n).set_index('Feature')[['Importance']]
xgb_survey_top_features = xgb_survey_fi.head(top_n).set_index('Feature')[['Importance']]

# Combine them into a single DataFrame for comparison
comparison_table = pd.concat([rf_top_features, xgb_admin_top_features, xgb_survey_top_features], axis=1)
comparison_table.columns = ['Random Forest Importance', 'XGBoost Importance', 'Survey-Enhanced XGBoost Importance']

print(f"Top {top_n} Features Comparison (Importance Scores):")
display(comparison_table.fillna(0).round(4))

Top 5 Features Comparison (Importance Scores):


,Random Forest Importance,XGBoost Importance,Survey-Enhanced XGBoost Importance
Feature,,,
GPA_2,0.2793,0.2236,0.0911
DFW_RATE_1,0.2102,0.3965,0.2867
GPA_1,0.1918,0.1619,0.0758
DFW_RATE_2,0.1172,0.1060,0.0560
UNITS_ATTEMPTED_2,0.1014,0.0624,0.0293


## 13. Survey-Enhanced Feature Group Importance

For the survey-enhanced XGBoost model, individual features are less important than the broader question: **What type of information is the model using?**

This section groups features into broad institutional categories.

In [ ]:
academic_prefixes = ['HS_GPA', 'HS_MATH_GPA', 'HS_ENGL_GPA', 'GPA_', 'DFW_RATE_',
                     'DFW_UNITS_', 'UNITS_ATTEMPTED_', 'UNITS_COMPLETED_', 'GRADE_POINTS_']
demo_prefixes = ['GENDER_', 'RACE_ETHNICITY_', 'FIRST_GEN_STATUS_', 'COLLEGE_']
survey_prefixes = ['Q', 'SURVEY_', 'NSSE_', 'ENGAGEMENT_', 'BELONGING_', 'SUPPORT_']
text_prefixes = ['TEXT_PC']

def assign_feature_group(col):
    col = str(col)
    if any(col.startswith(p) for p in academic_prefixes):
        return 'Academic performance'
    if any(col.startswith(p) for p in demo_prefixes):
        return 'Demographics'
    if any(col.startswith(p) for p in text_prefixes):
        return 'Survey/Text components'
    if any(col.startswith(p) for p in survey_prefixes):
        return 'Structured survey items'
    return 'Other / check'

if xgb_survey_fi is not None:
    xgb_survey_fi['Feature Group'] = xgb_survey_fi['Feature'].apply(assign_feature_group)
    group_importance = (xgb_survey_fi.groupby('Feature Group', as_index=False)['Importance']
                        .sum()
                        .sort_values('Importance', ascending=False))
    display(group_importance)
    fig = px.bar(group_importance, x='Feature Group', y='Importance',
                 title='Survey-Enhanced XGBoost: Total Importance by Feature Group')
    fig.update_layout(xaxis_tickangle=-20)
    fig.show()

,Feature Group,Importance
0,Academic performance,0.560945
2,Survey/Text components,0.321444
1,Demographics,0.117611


## 14. Classification Reports

The full classification report is helpful for checking whether a model is doing well for both the enrolled and departed classes.

In [ ]:
for name, info in trained_models.items():
    model = info['model']
    X_test = info['X_test']
    y_test = info['y_test']
    pred = model.predict(X_test)

    print('' + '=' * 90)
    print(name)
    print('=' * 90)
    print(classification_report(y_test, pred, target_names=['Enrolled', 'Departed'], zero_division=0))

Regularized Logistic
              precision    recall  f1-score   support

    Enrolled       0.90      0.99      0.94      4516
    Departed       0.84      0.42      0.56       820

    accuracy                           0.90      5336
   macro avg       0.87      0.70      0.75      5336
weighted avg       0.89      0.90      0.88      5336

Decision Tree
              precision    recall  f1-score   support

    Enrolled       0.95      0.87      0.91      4516
    Departed       0.52      0.75      0.61       820

    accuracy                           0.85      5336
   macro avg       0.73      0.81      0.76      5336
weighted avg       0.88      0.85      0.86      5336

Random Forest
              precision    recall  f1-score   support

    Enrolled       0.95      0.95      0.95      4516
    Departed       0.71      0.70      0.71       820

    accuracy                           0.91      5336
   macro avg       0.83      0.82      0.83      5336
weighted avg       0.91  

## 15. Equity Reports

In [ ]:
def evaluate_subgroup_classifier(name, model_obj, X_subgroup, y_subgroup, feature_set):
    """
    Evaluates a binary classifier's performance on a specific subgroup.
    Handles re-scaling of subgroup data if the model was trained on scaled features.
    """
    # Check if the subgroup data is empty
    if y_subgroup.empty:
        print(f"No data for subgroup for model {name}. Skipping evaluation.")
        return None

    X_subgroup_processed = X_subgroup

    # For models that were trained on scaled data (Logistic Regression, Random Forest),
    # the subgroup data must also be scaled using the same pre-fitted scaler.
    if name in ['Regularized Logistic', 'Random Forest']:
        # Ensure the input is a DataFrame for scaler.transform if it's not already.
        # X_subgroup should be a DataFrame if it came from X_test_admin or X_test_survey.loc[indices]
        X_subgroup_processed = scaler.transform(X_subgroup)
    # For XGBoost models, they typically handle scaling internally or don't require it explicitly.

    # Make predictions and probability estimates
    pred = model_obj.predict(X_subgroup_processed)
    prob = model_obj.predict_proba(X_subgroup_processed)[:, 1]

    # Calculate performance metrics
    return {
        'Model': name,
        'Feature Set': feature_set,
        'Subgroup Size': len(y_subgroup),
        'Subgroup Departure Rate': y_subgroup.mean(),
        'Accuracy': accuracy_score(y_subgroup, pred),
        'Precision': precision_score(y_subgroup, pred, zero_division=0),
        'Recall': recall_score(y_subgroup, pred, zero_division=0),
        'F1 Score': f1_score(y_subgroup, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_subgroup, prob),
        'Avg Precision': average_precision_score(y_subgroup, prob),
        'Brier Score': brier_score_loss(y_subgroup, prob),
        'Log Loss': log_loss(y_subgroup, prob)
    }

# --- Configure the demographic subgroup for comparison ---
# IMPORTANT: Ensure the chosen feature column exists in the original 'test_df'
# Examples:
# - 'GENDER_Female' (1.0 for female, 0.0 for male)
# - 'RACE_ETHNICITY_White' (1.0 for white, 0.0 for other)
subgroup_feature_col = 'GENDER_Female'
subgroup_filter_value = 1.0 # Change to 0.0 for male students, or other values for other groups

print(f"--- Model Performance for Subgroup: {subgroup_feature_col} == {subgroup_filter_value} ---")

# Validate the subgroup feature exists in test_df
if subgroup_feature_col not in test_df.columns:
    print(f"Error: Subgroup feature '{subgroup_feature_col}' not found in the original test_df.")
else:
    # Get the indices of the rows belonging to the specified subgroup in the full test_df
    subgroup_test_indices = test_df[test_df[subgroup_feature_col] == subgroup_filter_value].index

    if subgroup_test_indices.empty:
        print(f"No data found for subgroup '{subgroup_feature_col}' == {subgroup_filter_value}. Please choose a different subgroup or value.")
    else:
        subgroup_results = []
        for name, info in trained_models.items():
            model_obj = info['model']
            feature_set = info['feature_set']

            # Select the appropriate X_test based on the model's feature set
            if feature_set == 'Administrative':
                # X_test_admin only contains numeric features, but we filter its rows
                # based on indices from test_df that include demographic info.
                X_base_test_for_model = X_test_admin
            elif feature_set == 'Survey-enhanced':
                # X_test_survey contains both numeric and demographic features
                X_base_test_for_model = X_test_survey
            else:
                print(f"Warning: Unknown feature set '{feature_set}' for model '{name}'. Skipping evaluation.")
                continue

            # Filter the model's specific X_test and the global y_test for the subgroup
            X_subgroup_filtered = X_base_test_for_model.loc[subgroup_test_indices]
            y_subgroup_filtered = y_test.loc[subgroup_test_indices]

            # Evaluate model performance on this filtered subgroup data
            result = evaluate_subgroup_classifier(
                name, model_obj, X_subgroup_filtered, y_subgroup_filtered, feature_set
            )
            if result:
                subgroup_results.append(result)

        # Display the results in a DataFrame
        if subgroup_results:
            subgroup_results_df = pd.DataFrame(subgroup_results).set_index('Model')
            print("\n--- Subgroup Model Performance Metrics ---")
            display(subgroup_results_df.round(4))
        else:
            print("No subgroup evaluation results were generated.")

print("\nComparison complete.")

--- Model Performance for Subgroup: GENDER_Female == 1.0 ---

--- Subgroup Model Performance Metrics ---


,Feature Set,Subgroup Size,Subgroup Departure Rate,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Avg Precision,Brier Score,Log Loss
Model,,,,,,,,,,,
Regularized Logistic,Administrative,3261,0.1435,0.8620,0.5171,0.5812,0.5473,0.8097,0.6122,0.1136,0.4966
Decision Tree,Administrative,3261,0.1435,0.8592,0.5068,0.7179,0.5942,0.8467,0.6925,0.1067,0.3873
Random Forest,Administrative,3261,0.1435,0.6912,0.2870,0.7756,0.4189,0.8002,0.5129,0.2067,0.6326
XGBoost,Administrative,3261,0.1435,0.9034,0.6558,0.6880,0.6715,0.8575,0.7327,0.1109,0.3905
Survey-Enhanced XGBoost,Survey-enhanced,3261,0.1435,0.9095,0.6893,0.6731,0.6811,0.8625,0.7241,0.1067,0.3795



Comparison complete.


### Comprehensive Subgroup Performance Comparison

This section systematically evaluates each model's performance across all identified demographic subgroups. The results are aggregated into a single table for easy comparison.

In [ ]:
all_subgroup_results = []

# --- User selection: Choose ONE demographic variable prefix and ONE model ---
# Demographic Variable Prefix (e.g., 'GENDER_', 'RACE_ETHNICITY_', 'FIRST_GEN_STATUS_')
selected_demographic_variable_prefix = 'FIRST_GEN_STATUS_'
# Model Name (e.g., 'Regularized Logistic', 'Random Forest', 'XGBoost', 'Survey-Enhanced XGBoost')
selected_model_name = 'Random Forest'

print(f"Evaluating performance for model '{selected_model_name}' across subgroups of: {selected_demographic_variable_prefix}")

# Identify demographic columns from test_df belonging to the selected prefix
selected_demographic_cols = [col for col in test_df.columns
                             if col.startswith(selected_demographic_variable_prefix)
                             and col != selected_demographic_variable_prefix.rstrip('_')]

if not selected_demographic_cols:
    print(f"No demographic columns found for prefix '{selected_demographic_variable_prefix}'. Please check the prefix.")
else:
    for col in selected_demographic_cols:
        # We are interested in subgroups where the feature is '1.0' (e.g., GENDER_Female == 1)
        value = 1.0

        print(f"\n--- Subgroup: {col} == {value} ---")
        subgroup_test_indices = test_df[test_df[col] == value].index

        if subgroup_test_indices.empty:
            print(f"No data found for subgroup '{col}' == {value}. Skipping.")
            continue

        # Process only the selected model
        if selected_model_name in trained_models:
            name = selected_model_name
            info = trained_models[name]
            model_obj = info['model']
            feature_set = info['feature_set']

            if feature_set == 'Administrative':
                X_base_test_for_model = X_test_admin
            elif feature_set == 'Survey-enhanced':
                X_base_test_for_model = X_test_survey
            else:
                print(f"Warning: Unknown feature set '{feature_set}' for model '{name}'. Skipping evaluation.")
                continue

            X_subgroup_filtered = X_base_test_for_model.loc[subgroup_test_indices]
            y_subgroup_filtered = y_test.loc[subgroup_test_indices]

            result = evaluate_subgroup_classifier(
                name, model_obj, X_subgroup_filtered, y_subgroup_filtered, feature_set
            )
            if result:
                result['Demographic Feature'] = col
                result['Subgroup Value'] = value
                all_subgroup_results.append(result)
        else:
            print(f"Warning: Model '{selected_model_name}' not found in trained_models. Please check the model name.")

    # Create a DataFrame from all results for the selected model
    if all_subgroup_results:
        # Filter to include only the relevant columns for direct comparison
        # 'Demographic Feature' will be consistent, 'Subgroup Value' is always 1.0
        # So we can set 'Demographic Feature' as the index and keep 'Model' as a column or drop it if it's fixed
        filtered_subgroup_df = pd.DataFrame(all_subgroup_results)

        # Drop the 'Subgroup Value' and 'Model' columns for a cleaner table as they are fixed/redundant here
        filtered_subgroup_df = filtered_subgroup_df.drop(columns=['Subgroup Value', 'Model'])
        filtered_subgroup_df = filtered_subgroup_df.set_index('Demographic Feature')

        print(f"\n--- Performance Metrics for '{selected_model_name}' by Subgroup of '{selected_demographic_variable_prefix}' ---")
        display(filtered_subgroup_df.round(4))
    else:
        print("No subgroup evaluation results were generated for the selected model and demographic variable.")

print("\nComparison complete.")

Evaluating performance for model 'Random Forest' across subgroups of: FIRST_GEN_STATUS_

--- Subgroup: FIRST_GEN_STATUS_Continuing Generation == 1.0 ---

--- Subgroup: FIRST_GEN_STATUS_First Generation == 1.0 ---

--- Subgroup: FIRST_GEN_STATUS_Unknown == 1.0 ---

--- Performance Metrics for 'Random Forest' by Subgroup of 'FIRST_GEN_STATUS_' ---


,Feature Set,Subgroup Size,Subgroup Departure Rate,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Avg Precision,Brier Score,Log Loss
Demographic Feature,,,,,,,,,,,
FIRST_GEN_STATUS_Continuing Generation,Administrative,3312,0.1341,0.6754,0.2522,0.7230,0.3739,0.7690,0.4457,0.2083,0.6345
FIRST_GEN_STATUS_First Generation,Administrative,1537,0.1731,0.6402,0.3153,0.9211,0.4698,0.8631,0.6058,0.2381,0.7219
FIRST_GEN_STATUS_Unknown,Administrative,487,0.2259,0.6181,0.3582,0.8727,0.5079,0.8119,0.5834,0.2510,0.7531



Comparison complete.


Model consistency can be measured by considering the standard deviation of each metric across subgroups:

In [ ]:
all_subgroup_results_df = pd.DataFrame(all_subgroup_results)

# Define the metrics for which to calculate standard deviation
metrics_to_std = [
    'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC',
    'Avg Precision', 'Brier Score', 'Log Loss'
]

# Group by model and calculate the standard deviation for each metric
std_dev_by_model = all_subgroup_results_df.groupby('Model')[metrics_to_std].std()

print("\n--- Standard Deviation of Performance Metrics by Model Across Subgroups ---")
display(std_dev_by_model.round(4))

### Re-evaluating Subgroup Performance for All Models to Calculate Comprehensive Standard Deviations

To compute the standard deviation of performance metrics for **all four models** across the chosen demographic variable's subgroups, we need to gather all the subgroup results first. The previous `all_subgroup_results` only contained data for the `Survey-Enhanced XGBoost` model. The following cell will re-run the evaluation for all models against the same `RACE_ETHNICITY_` subgroups.

In [ ]:
all_models_subgroup_results = []

# Keep the same demographic variable prefix for consistency with the previous analysis
selected_demographic_variable_prefix = 'RACE_ETHNICITY_'

print(f"Re-evaluating performance for all models across subgroups of: {selected_demographic_variable_prefix}")

# Identify demographic columns from test_df belonging to the selected prefix
selected_demographic_cols = [col for col in test_df.columns
                             if col.startswith(selected_demographic_variable_prefix)
                             and col != selected_demographic_variable_prefix.rstrip('_')]

if not selected_demographic_cols:
    print(f"No demographic columns found for prefix '{selected_demographic_variable_prefix}'. Please check the prefix.")
else:
    for col in selected_demographic_cols:
        # We are interested in subgroups where the feature is '1.0' (e.g., GENDER_Female == 1)
        value = 1.0

        # print(f"\n--- Subgroup: {col} == {value} ---") # Uncomment for detailed progress
        subgroup_test_indices = test_df[test_df[col] == value].index

        if subgroup_test_indices.empty:
            print(f"No data found for subgroup '{col}' == {value}. Skipping.")
            continue

        # Iterate through ALL trained models
        for name, info in trained_models.items():
            model_obj = info['model']
            feature_set = info['feature_set']

            if feature_set == 'Administrative':
                X_base_test_for_model = X_test_admin
            elif feature_set == 'Survey-enhanced':
                X_base_test_for_model = X_test_survey
            else:
                print(f"Warning: Unknown feature set '{feature_set}' for model '{name}'. Skipping evaluation.")
                continue

            X_subgroup_filtered = X_base_test_for_model.loc[subgroup_test_indices]
            y_subgroup_filtered = y_test.loc[subgroup_test_indices]

            result = evaluate_subgroup_classifier(
                name, model_obj, X_subgroup_filtered, y_subgroup_filtered, feature_set
            )
            if result:
                result['Demographic Feature'] = col
                result['Subgroup Value'] = value
                all_models_subgroup_results.append(result)


print("\nFinished collecting subgroup performance data for all models.")

Re-evaluating performance for all models across subgroups of: RACE_ETHNICITY_

Finished collecting subgroup performance data for all models.


### Standard Deviation of Performance Metrics Across Subgroups for All Models

Now that we have collected the performance metrics for all models across each demographic subgroup, we can calculate the standard deviation for each metric. This will highlight how consistently each model performs across these different population segments.

In [ ]:
all_models_subgroup_results_df = pd.DataFrame(all_models_subgroup_results)

# Define the metrics for which to calculate standard deviation
metrics_to_std = [
    'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC',
    'Avg Precision', 'Brier Score', 'Log Loss'
]

# Group by model and calculate the standard deviation for each metric
std_dev_by_model_all = all_models_subgroup_results_df.groupby('Model')[metrics_to_std].std()

# Find the model with the lowest standard deviation for each metric
lowest_std_model = std_dev_by_model_all.idxmin()

# Create a new DataFrame for the 'Lowest Std Dev Model' row
lowest_std_row = pd.DataFrame([lowest_std_model], index=['Lowest Std Dev Model'])

# Concatenate the new row to the existing standard deviation DataFrame
std_dev_by_model_all_with_lowest = pd.concat([std_dev_by_model_all, lowest_std_row])

print("\n--- Standard Deviation of Performance Metrics by Model Across Subgroups ---")
display(std_dev_by_model_all_with_lowest.round(4))


--- Standard Deviation of Performance Metrics by Model Across Subgroups ---


,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Avg Precision,Brier Score,Log Loss
Decision Tree,0.028297,0.09221,0.122051,0.074568,0.079319,0.095753,0.013812,0.028463
Random Forest,0.057875,0.081817,0.138386,0.096444,0.065807,0.155391,0.029829,0.079347
Regularized Logistic,0.038048,0.156302,0.159418,0.141283,0.090204,0.166882,0.02673,0.120517
Survey-Enhanced XGBoost,0.013959,0.106717,0.106485,0.078149,0.068545,0.109299,0.011973,0.027449
XGBoost,0.018038,0.083703,0.121429,0.078232,0.069753,0.109742,0.010981,0.024623
Lowest Std Dev Model,Survey-Enhanced XGBoost,Random Forest,Survey-Enhanced XGBoost,Decision Tree,Random Forest,Decision Tree,XGBoost,XGBoost


## 15. Recommendation Framework for Higher Education

Use the model comparison results to choose a model based on the institutional use case.

| Institutional Priority | Recommended Model | Rationale |
|---|---|---|
| Transparency and stakeholder trust | Regularized Logistic Regression | Most explainable and easiest to defend |
| Nonlinear patterns with moderate interpretability | Random Forest | Captures interactions while remaining more explainable than boosting |
| Strong administrative-data prediction | XGBoost | Often improves performance on structured institutional data |
| Best use of student voice and survey data | Survey-Enhanced XGBoost | Combines academic records, demographics, survey responses, and compressed text features |

### Important caution

The survey-enhanced model may perform better because it contains richer data. However, richer data also raises higher ethical and governance obligations. Before deployment, review consent scope, privacy risk, transparency, and whether the intervention will support students rather than label or punish them.

## 16. Export Results

The final comparison table can be saved for reporting or included in a model card.

In [ ]:
# Save the trained calibrated model to a file using pickle
filename = f'{model_filepath}Survey_xgb_model.pkl'
pickle.dump(xgb_survey, open(filename, 'wb'))

print(f"Survey Enhanced xgb model saved to: {filename}")

## 17. Summary

In this notebook, you created an advanced systematic comparison framework that:

1. Reproduces the original 4.1 comparison across Logistic Regression, Random Forest, and XGBoost
2. Adds the Module 8 survey-enhanced XGBoost model
3. Preserves each model's hyperparameters in a reusable registry
4. Compares models using standard classification metrics
5. Visualizes performance using grouped bars, ROC curves, Precision-Recall curves, and confusion matrices
6. Compares practical deployment trade-offs using a radar chart
7. Interprets feature importance, including feature-group importance for the survey-enhanced model

The central institutional question is no longer simply **Which model wins?** It is:

> Which model provides the right balance of predictive performance, ethical defensibility, stakeholder trust, and actionable student support?